In [ ]:
import pandas as pd
import geopandas as gpd
from pygris import counties
from census import Census

In [ ]:
CENSUS_API_KEY = "86e9f1f0de0f6f61dedbef79dbe6f750a5cad585"
YEAR = 2024  # Latest ACS 5-year estimates as of April 2026

In [ ]:
# census variable codes
variables = [
    'B01003_001E', # Total Pop
    'B19013_001E', # Median Income
    'B01002_001E', # Median Age
    'B19001_017E', # Households $200k+
    'B19001_001E', # Total households
    'B15003_022E', # Bachelors degree (age 25+)
    'B15003_001E', # Total education population
    'B03002_003E', # Total white population
    'B23025_002E', # Labor force
    'B23025_005E' # Unemployed population
]

In [ ]:
def get_county_stats_and_shapes():
    print("Downloading US County shapefiles...")
    # 'cb=True' uses Generalized Cartographic Boundaries
    # resolution='5m' is 1:5,000,000 
    usa_counties_gdf = counties(cb=True, resolution='5m', year=YEAR)
    
    # retrieve Census data
    print("Fetching population and income from Census API...")
    c = Census(CENSUS_API_KEY)
    
    # Pull data for all counties in all states
    census_data = c.acs5.get(variables, {'for': 'county:*', 'in': 'state:*'}, year=YEAR)
    
    # Convert to DataFrame
    df = pd.DataFrame(census_data)
    
    # Create the standard 5-digit FIPS code
    df['GEOID'] = df['state'] + df['county']
    
    # Convert all Census variables to numeric
    for var in variables:
        df[var] = pd.to_numeric(df[var], errors='coerce')

    # Rename columns for clarity
    df = df.rename(columns={
        'B01003_001E': 'population',
        'B19013_001E': 'median_income',
        'B01002_001E': 'median_age'
    })

    # Feature engineering
    # high income ration
    df['pct_high_income'] = df['B19001_017E'] / df['B19001_001E']
    
    # bachelors pct
    df['pct_bachelors'] = df['B15003_022E'] / df['B15003_001E']
    
    # white pct
    df['pct_white'] = df['B03002_003E'] / df['population']
    
    # unemployment_rate
    df['unemployment_rate'] = df['B23025_005E'] / df['B23025_002E']
    
    # List of engineered features to keep
    keep_cols = [
        'GEOID', 'population', 'median_income', 
        'median_age', 'pct_high_income', 'pct_bachelors', 
        'pct_white', 'unemployment_rate'
    ]

    # Merge shapefile with stats
    final_gdf = usa_counties_gdf.merge(df[keep_cols], on='GEOID')

    # Add population density
    final_gdf['density'] = 10000 * final_gdf['population'] / final_gdf['ALAND']
    
    return final_gdf

In [ ]:
counties_gdf = get_county_stats_and_shapes()

In [ ]:
counties_gdf.head()

In [ ]:
# drop Puerto Rico counties since they aren't included in the fandom data
counties_gdf = counties_gdf[counties_gdf["STUSPS"] != "PR"].reset_index(drop=True)

In [ ]:
# correct missing median incomes with mean value of the state
state_means = (
    counties_gdf[counties_gdf["median_income"] > 0]
    .groupby("STUSPS")["median_income"]
    .mean()
    .round()
)

invalid_mask = counties_gdf["median_income"] < 0

counties_gdf.loc[invalid_mask, "median_income"] = counties_gdf.loc[invalid_mask, "STUSPS"].map(state_means)

In [ ]:
def plot_interactive_counties(gdf, column_to_plot='median_income'):
    # Generate Interactive Map

    tooltip_columns = ['GEOID', 'NAME', 'population'] + [column_to_plot]

    m = gdf.explore(
        column=column_to_plot, 
        cmap='YlGnBu',
        tiles='CartoDB Positron',
        tooltip=tooltip_columns,
        popup=True, # Clicking shows all attributes
        legend_kwds={'caption': f'US County {column_to_plot.replace("_", " ").title()}'},
        style_kwds={'weight': 0.5, 'color': 'white'} # Thin white borders
    )

    return m

In [ ]:
plot_interactive_counties(counties_gdf)

In [ ]:
# select final data
counties_full = counties_gdf[
    ['GEOID', 'NAME', 'STUSPS', 'STATE_NAME', 'population',
     'density', 'median_income', 'median_age', 'pct_high_income',
     'pct_bachelors', 'pct_white', 'unemployment_rate', 'geometry']
].rename(columns={
    'GEOID': 'geoid',
    'NAME': 'county_name',
    'STUSPS': 'state_code',
    'STATE_NAME': 'state_name'
})

In [ ]:
counties_full.shape

In [ ]:
counties_full.head()

In [ ]:
counties_full.to_csv('../data/processed_data/county_node.csv', index=False)